# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**What this notebook is.** Before I model anything, I test whether the stories people tell about search content show up in the starter file. Every test has the claim, a table with visible n's, and a verdict (CONFIRMED / OPPOSITE / MIXED / FALSE). A "no" is a valid result. All numbers are **observed** on one 30,000-page snapshot (32 clients); nothing here is causal, and nothing here says anything about how Google ranks pages.

Rules I follow: heavy tails are handled with medians, ranks and pooled (weighted) rates, never a mean of per-page rates; no verdict from a bucket under 50 pages; every result that could be a client effect gets a **within-client** re-check.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**What I expect, then check:** traffic fields are heavy-tailed, so a mean is a bad summary and a plain correlation will be driven by a few giant pages. Missing values also need a look: a zero or a blank can mean "not measured" rather than "nothing happened" (`avg_position = 0` means no position data; blank `word_count` is missing).

In [1]:
import os, sys, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"
if IN_COLAB:
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
os.makedirs("work/figures", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages x {df.shape[1]} columns, {df['client_id'].nunique()} clients")

cols = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "word_count", "content_age_days"]
q = df[cols].quantile([.5, .9, .99]).T
q.columns = ["median", "p90", "p99"]
q["max"] = df[cols].max()
q["missing_%"] = (100 * df[cols].isna().mean()).round(1)
q["zero_%"] = (100 * (df[cols] == 0).mean()).round(1)
display(q.round(1))

for c in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    s = df[c].sort_values(ascending=False)
    print(f"{c}: the top 1% of pages (300) hold {s.head(300).sum() / s.sum():.1%} of the total")

print("\nword_count missing share by content type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
ax[0].hist(np.log10(df["impressions_90d"] + 1), bins=50, color="#2b6cb0")
ax[0].set_title("90-day impressions per page (log10 scale)", fontsize=10)
ax[0].set_xlabel("log10(impressions + 1)"); ax[0].set_ylabel("pages")
ax[1].hist(np.log10(df["clicks_90d"] + 1), bins=50, color="#2b6cb0")
ax[1].set_title("90-day clicks per page (log10 scale)", fontsize=10)
ax[1].set_xlabel("log10(clicks + 1)")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("work/figures/fig_signal_distributions.png", dpi=150); plt.show()

30,000 pages x 44 columns, 32 clients


,median,p90,p99,max,missing_%,zero_%
impressions_90d,731.0,12136.4,73505.8,517715.0,0.0,0.0
clicks_90d,1.0,32.0,253.0,4178.0,0.0,44.0
sessions_90d,7.0,88.0,451.0,4345.0,0.0,0.0
avg_position,10.8,36.8,69.9,245.0,0.0,4.0
word_count,2877.0,5327.0,7292.0,9546.0,25.7,0.0
content_age_days,236.0,463.0,537.0,564.0,0.0,0.0


impressions_90d: the top 1% of pages (300) hold 24.9% of the total
clicks_90d: the top 1% of pages (300) hold 35.0% of the total
sessions_90d: the top 1% of pages (300) hold 21.5% of the total

word_count missing share by content type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283


**Reading.** Impressions and clicks are strongly right-skewed: the median page has far fewer impressions than the mean, and the top 1% of pages hold roughly a quarter of impressions and about a third of clicks. So: **medians, rank correlation (Spearman) and pooled rates** from here on. About a quarter of pages have no `word_count`, and the missing share depends on content type (keyword articles only), so I will never fill it with a blind zero in a test.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Test 1: "Higher position means a higher CTR"
**Slice:** pages with at least 100 impressions in 90 days and a real position of 1-20. **Rate = total clicks / total impressions** in the band (not a mean of page CTRs). Floor: at least 50 pages per band.

In [2]:
# ---- Test 1: position band vs CTR (pooled rate)
v = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
v["band"] = pd.cut(v["avg_position"], [0, 3, 6, 10, 15, 20], labels=["1-3", "4-6", "7-10", "11-15", "16-20"])
g = v.groupby("band", observed=True).agg(pages=("content_id", "size"), clicks=("clicks_90d", "sum"), impressions=("impressions_90d", "sum"))
g["CTR_%"] = (100 * g["clicks"] / g["impressions"]).round(3)
assert (g["pages"] >= 50).all()
display(g)

# robustness: the same table on the LATER 30 days only, and the band medians of page-level CTR
l = v[v["impressions_last_30d"] >= 100].copy()
gl = l.groupby("band", observed=True).agg(pages=("content_id", "size"), clicks=("clicks_last_30d", "sum"), impressions=("impressions_last_30d", "sum"))
gl["CTR_%"] = (100 * gl["clicks"] / gl["impressions"]).round(3)
print("Re-run on a different slice (later 30 days, pages with >=100 impressions then):")
display(gl)

# is the order the same inside each client? count clients where band 1-6 beats band 7-10 on pooled CTR
def pooled(d):
    return 100 * d["clicks_90d"].sum() / d["impressions_90d"].sum()
v["top6"] = v["avg_position"] <= 6
wc = v[v["avg_position"] <= 10].groupby(["client_id", "top6"]).agg(n=("content_id", "size"), c=("clicks_90d", "sum"), i=("impressions_90d", "sum")).reset_index()
wc["ctr"] = 100 * wc["c"] / wc["i"]
piv = wc.pivot(index="client_id", columns="top6", values="ctr").dropna()
npg = wc.pivot(index="client_id", columns="top6", values="n").dropna()
ok = (npg >= 30).all(axis=1)
piv = piv[ok]
print(f"Within clients (>=30 pages in both groups, {len(piv)} clients): positions 1-6 out-earn 7-10 in {(piv[True] > piv[False]).sum()} of {len(piv)}")

,pages,clicks,impressions,CTR_%
band,,,,
1-3,555,36914,7554986,0.489
4-6,3436,204003,50742531,0.402
7-10,5224,107219,38536973,0.278
11-15,3524,47267,13298455,0.355
16-20,2352,32116,9475953,0.339


Re-run on a different slice (later 30 days, pages with >=100 impressions then):


,pages,clicks,impressions,CTR_%
band,,,,
1-3,377,11414,2002083,0.570
4-6,2787,60118,14495298,0.415
7-10,3963,33764,11602978,0.291
11-15,2572,14276,3513402,0.406
16-20,1739,9795,2314193,0.423


Within clients (>=30 pages in both groups, 13 clients): positions 1-6 out-earn 7-10 in 9 of 13


**Verdict #1: MIXED.** The big picture holds: positions 1-6 earn a higher pooled CTR than 7-10, so *top-of-page is worth more than the bottom of page one* (and 9 of the 13 clients with enough pages show the same ordering). But the ladder is not clean: 11-15 and 16-20 earn **more** than 7-10, and even positions 1-3 average under 0.5% here (this snapshot's CTR is a pooled, 90-day, all-query average, not a classic branded-click curve). **In practice:** use position as a *band* to compare a page against its own peers, never as a smooth "move up one spot, gain X clicks" rule. That is exactly why my capstone label compares each page with its band's typical CTR.

### Test 2: "Longer pages get more traffic"
**Slice:** pages with a word count and at least 100 impressions. **Why this test is risky:** word count is missing for about a quarter of pages and clients write differently, so a pooled correlation can simply be a client effect. I run it pooled first, then **within each client** (clients with at least 50 qualifying pages).

In [3]:
# ---- Test 2: word count vs impressions (rank correlation, pooled and within-client)
w = df[(df["word_count"] > 0) & (df["impressions_90d"] >= 100)].copy()
w["length quartile"] = pd.qcut(w["word_count"], 4, labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"])
t = w.groupby("length quartile", observed=True).agg(pages=("content_id", "size"), median_words=("word_count", "median"),
                                                    median_impressions=("impressions_90d", "median"))
display(t)
rho_pooled = spearmanr(w["word_count"], w["impressions_90d"])[0]
print(f"Pooled Spearman (word_count vs impressions_90d): {rho_pooled:.3f}")

per = {}
for cid, d in w.groupby("client_id"):
    if len(d) >= 50:
        per[cid] = spearmanr(d["word_count"], d["impressions_90d"])[0]
per = pd.Series(per)
print(f"Within-client Spearman, {len(per)} clients with >=50 pages: median {per.median():.3f}, "
      f"positive in {(per > 0).sum()} of {len(per)}, range {per.min():.2f} to {per.max():.2f}")

print("\nPooled Spearman by content type:")
print(w.groupby("content_type").apply(lambda d: pd.Series({"pages": len(d), "rho": spearmanr(d["word_count"], d["impressions_90d"])[0]})).round(3).to_string())

,pages,median_words,median_impressions
length quartile,,,
Q1 shortest,3864,1708.0,970.0
Q2,3873,2758.0,2025.0
Q3,3851,3173.0,2001.0
Q4 longest,3863,5477.0,3989.0


Pooled Spearman (word_count vs impressions_90d): 0.251
Within-client Spearman, 18 clients with >=50 pages: median 0.001, positive in 9 of 18, range -0.34 to 0.38

Pooled Spearman by content type:
                      pages    rho
content_type                      
comparison article    366.0 -0.228
feedly article        352.0  0.229
keyword article     14733.0  0.249


**Verdict #2: FALSE (as a general rule).** Pooled across all clients, longer pages do have more impressions (Spearman about +0.25; the longest quartile's median is several times the shortest's). But **inside a single client the relationship vanishes**: the median within-client correlation is about zero and it is positive in only about half of the clients. The pooled link is mostly "some clients publish longer pages and also have more search visibility", i.e. a client effect. **In practice:** do not tell a writer to "make it longer" on this evidence.

### Test 3: "Older pages decay: they are the ones losing traffic"
**Slice:** pages with at least 100 impressions in both 30-day windows (so a decline is a real decline, not a page vanishing). **Outcome:** later-window impressions below 80% of the earlier window ("down"). This is a descriptive outcome, not a model label. Floor: 50 pages per age tier.

In [4]:
# ---- Test 3: content age vs decline between the two 30-day windows
d3 = df[(df["impressions_prev_30d"] >= 100) & (df["impressions_last_30d"] >= 100)].copy()
d3["ratio"] = d3["impressions_last_30d"] / d3["impressions_prev_30d"]
d3["down"] = d3["ratio"] < 0.8
order = ["31-90", "91-180", "181-365", "365+"]
t3 = d3.groupby("age_tier").agg(pages=("content_id", "size"), share_down=("down", "mean"), median_ratio=("ratio", "median")).loc[order]
t3["share_down"] = (100 * t3["share_down"]).round(1)
t3["median_ratio"] = t3["median_ratio"].round(3)
t3 = t3.rename(columns={"share_down": "% down (<80% of earlier impressions)", "median_ratio": "median later/earlier impressions"})
assert (t3["pages"] >= 50).all()
display(t3)

print("Same table for keyword articles only (the dominant type), to rule out a content-type mix effect:")
k = d3[d3["content_type"] == "keyword article"].groupby("age_tier")["down"].agg(["size", "mean"]).loc[order]
k["mean"] = (100 * k["mean"]).round(1)
display(k.rename(columns={"size": "pages", "mean": "% down"}))

# how many clients even have both young and old pages? (limits any within-client check)
has = d3.groupby("client_id")["age_tier"].agg(lambda s: set(s))
both = [c for c, s in has.items() if "365+" in s and "91-180" in s]
print(f"Clients with both 91-180 and 365+ pages in this slice: {len(both)} of {d3['client_id'].nunique()}")

,pages,% down (<80% of earlier impressions),median later/earlier impressions
age_tier,,,
31-90,190,67.9,0.575
91-180,5871,65.0,0.641
181-365,5662,56.4,0.742
365+,3892,41.0,0.865


Same table for keyword articles only (the dominant type), to rule out a content-type mix effect:


,pages,% down
age_tier,,
31-90,189,68.3
91-180,5799,65.2
181-365,5547,56.6
365+,3892,41.0


Clients with both 91-180 and 365+ pages in this slice: 4 of 29


**Verdict #3: OPPOSITE.** The older the page, the *less* often it is down: roughly two thirds of the youngest tier fell more than 20%, against about 40% of pages older than a year. The same pattern holds inside keyword articles alone. The likely story is not "young pages are bad": newly published pages are still settling after launch, while old pages that survived have stabilised. **Caveat (directional only):** age is tangled up with *which clients published when*, and only a handful of clients have both young and old pages, so I cannot separate age from client with this snapshot. **In practice:** do not use "page is old" as a decay rule; ask "what changed recently?" instead.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**The flag family:** FlyRank's app flags pages that "need a CTR fix" or are "quick wins". These are hand-written rules (I do not have the flags themselves; the data ships observable signals only). Both rest on an assumption I can test:

1. **"Needs CTR fix" assumes a CTR shortfall is a stable problem**: a page that earns far less than its position normally earns today will still do so later (otherwise a review is a waste).
2. **"Quick win (striking distance)" assumes pages ranked about 11-20 are the ones with room to move.** If that were true, those bands should look different from the rest in momentum.

Both use the pooled position bands from Test 1. "Low" = page CTR below half of the band's typical (pooled) CTR in the same window. Slice: pages with at least 100 impressions in both windows and a position of 1-20.

In [5]:
# ---- Flag-linked test: do the rule assumptions hold?
e = d3[(d3["avg_position"] > 0) & (d3["avg_position"] <= 20)].copy()
e["band"] = pd.cut(e["avg_position"], [0, 3, 6, 10, 15, 20], labels=["1-3", "4-6", "7-10", "11-15", "16-20"])
e["ctr_prev"] = 100 * e["clicks_prev_30d"] / e["impressions_prev_30d"]
e["ctr_last"] = 100 * e["clicks_last_30d"] / e["impressions_last_30d"]
typ_prev = e.groupby("band", observed=True).apply(lambda g: 100 * g["clicks_prev_30d"].sum() / g["impressions_prev_30d"].sum())
typ_last = e.groupby("band", observed=True).apply(lambda g: 100 * g["clicks_last_30d"].sum() / g["impressions_last_30d"].sum())
e["low_prev"] = e["ctr_prev"] < 0.5 * e["band"].map(typ_prev).astype(float)
e["low_last"] = e["ctr_last"] < 0.5 * e["band"].map(typ_last).astype(float)

print("Assumption 1: a CTR shortfall persists")
a1 = e.groupby("low_prev")["low_last"].agg(pages="size", still_low="mean")
a1["still_low"] = (100 * a1["still_low"]).round(1)
a1.index = ["earlier CTR normal for the band", "earlier CTR under half the band norm"]
display(a1.rename(columns={"still_low": "% low in the later window"}))
assert (a1["pages"] >= 50).all()

print("Assumption 2: striking-distance bands (11-20) show different momentum")
a2 = e.groupby("band", observed=True).agg(pages=("content_id", "size"), pct_down=("down", "mean"), median_ratio=("ratio", "median"))
a2["pct_down"] = (100 * a2["pct_down"]).round(1); a2["median_ratio"] = a2["median_ratio"].round(3)
display(a2.rename(columns={"pct_down": "% down", "median_ratio": "median later/earlier impressions"}))

# does it hold inside clients? share of clients (with >=30 pages in each group) where low-earlier pages are more often low-later
rows = []
for cid, d in e.groupby("client_id"):
    lo, hi = d[d["low_prev"]], d[~d["low_prev"]]
    if len(lo) >= 30 and len(hi) >= 30:
        rows.append(lo["low_last"].mean() - hi["low_last"].mean())
rows = pd.Series(rows)
print(f"Within clients ({len(rows)} with >=30 pages in both groups): low-earlier pages are MORE often low-later in {(rows > 0).sum()} of {len(rows)}; median gap {100 * rows.median():.0f} points")

Assumption 1: a CTR shortfall persists


,pages,% low in the later window
earlier CTR normal for the band,6128,23.3
earlier CTR under half the band norm,4983,62.7


Assumption 2: striking-distance bands (11-20) show different momentum


,pages,% down,median later/earlier impressions
band,,,
1-3,373,68.6,0.590
4-6,2748,56.4,0.743
7-10,3849,54.4,0.762
11-15,2474,56.7,0.740
16-20,1667,59.9,0.722


Within clients (14 with >=30 pages in both groups): low-earlier pages are MORE often low-later in 14 of 14; median gap 36 points


**Verdict (assumption 1, "a CTR shortfall persists"): CONFIRMED.** Pages earning under half their band's normal CTR in the earlier 30 days are still low in the later 30 days about **63%** of the time, against about **23%** for pages that looked normal. It also holds inside clients (14 of 14 clients with enough pages, median gap about 36 points), so it is not a client effect. This is the data-backed core of my capstone: a CTR gap is persistent enough that reviewing it is worth a person's time.

**Verdict (assumption 2, "striking-distance pages have room to move"): FALSE as stated.** From 4 to 20 the share of pages down stays within about 6 points (54-60%) and the median change is about the same; only the 1-3 band looks worse, and it has few pages. Nothing here says the 11-20 pages are special; I can only say the data does not show an extra effect. **In practice:** keep the CTR-gap flag, and treat "quick win by position" as an unproven assumption rather than a finding.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

- **Compare a page with its own position band, not with a universal CTR curve** (Test 1: the position ladder is mixed).
- **A real, persistent CTR gap is the one signal that clearly holds** (flag test 1: 63% vs 23% still low later), so a CTR-gap review list is a defensible decision-support tool.
- **Do not act on "page is long" or "page is old"** (Tests 2 and 3): the first is a client effect, the second points the opposite way from the usual story.

All of this is observed on one snapshot, directional, and not evidence of how Google works or of what a content change would cause.

In [6]:
# ---- one-screen summary of the verdicts, so the notebook ends with the answer
summary = pd.DataFrame([
    ["Higher position -> higher CTR", "MIXED", "top 1-6 > 7-10, but 11-20 earn more than 7-10"],
    ["Longer pages -> more traffic", "FALSE (client effect)", "pooled rho ~ +0.25, within-client median ~ 0"],
    ["Older pages decay more", "OPPOSITE", "youngest tier most often down; 365+ least"],
    ["Flag assumption: CTR shortfall persists", "CONFIRMED", "63% vs 23% still low later; holds within clients"],
    ["Flag assumption: 11-20 pages have room to move", "FALSE as stated", "momentum looks alike in every band"],
], columns=["signal", "verdict", "evidence in one line"])
display(summary.style.hide(axis="index"))

signal,verdict,evidence in one line
Higher position -> higher CTR,MIXED,"top 1-6 > 7-10, but 11-20 earn more than 7-10"
Longer pages -> more traffic,FALSE (client effect),"pooled rho ~ +0.25, within-client median ~ 0"
Older pages decay more,OPPOSITE,youngest tier most often down; 365+ least
Flag assumption: CTR shortfall persists,CONFIRMED,63% vs 23% still low later; holds within clients
Flag assumption: 11-20 pages have room to move,FALSE as stated,momentum looks alike in every band


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.